In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("/kaggle/input/datasets/karvysingh/dalisa/FCC_feed_data.xlsx",
                   sheet_name="Feed Data")

# remove the units row
df = df[pd.to_numeric(df["Sample No."], errors="coerce").notna()].copy()

df["Sample No."] = df["Sample No."].astype(int)

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c])

df.head()

,Sample No.,Sp. Gr,CCR,T.S,T.N,IBP,5 %,10 %,30 %,50 %,70 %,90 %,95 %,FBP,P,N,A
1,1,0.9245,0.40,2.93,1401,37,317,342,392,423,454,498,514,538,8.3,24.1,67.6
2,2,0.9960,1.55,4.35,200,42,409,428,458,477,494,514,523,537,7.8,25.3,66.8
3,3,0.8683,0.04,0.03,22,103,224,263,337,380,420,473,497,536,15.2,45.0,39.8
4,4,0.9136,0.04,1.54,329,111,276,318,368,391,407,427,436,460,14.8,15.5,70.5
5,5,0.9303,0.28,3.06,1161,169,257,320,402,438,474,519,535,574,11.2,23.3,65.6


In [2]:
features = [
    "Sp. Gr",
    "5 %",
    "10 %",
    "30 %",
    "50 %",
    "70 %",
    "90 %",
    "95 %"
]

targets = ["P", "N", "A"]

# published validation samples
val = df[df["Sample No."].between(17, 22)].copy()

# remaining samples for development
dev = df[~df["Sample No."].between(17, 22)].copy()

X = dev[features]
y = dev[targets]

X_val = val[features]
y_val = val[targets]

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size=16,
    random_state=42
)

scaler = MinMaxScaler(feature_range=(-1, 1))

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
X_val   = scaler.transform(X_val)

In [4]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

tf.keras.utils.set_random_seed(42)

def make_model(h1, h2):
    return Sequential([
        Dense(h1, activation="tanh", input_shape=(8,)),
        Dense(h2, activation="sigmoid"),
        Dense(1, activation="linear")
    ])

model_P = make_model(12, 13)
model_N = make_model(12, 12)
model_A = make_model(9, 9)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-09-27 07:49:36.244102: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
for model in [model_P, model_N, model_A]:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
        loss="mse"
    )

model_P.fit(X_train, y_train["P"],
            epochs=1000, verbose=0)

model_N.fit(X_train, y_train["N"],
            epochs=1000, verbose=0)

model_A.fit(X_train, y_train["A"],
            epochs=1000, verbose=0)

In [6]:
P = model_P.predict(X_val, verbose=0).ravel()
N = model_N.predict(X_val, verbose=0).ravel()
A = model_A.predict(X_val, verbose=0).ravel()

total = P + N + A

P = 100 * P / total
N = 100 * N / total
A = 100 * A / total

pred = np.column_stack([P, N, A])

In [7]:
result = pd.DataFrame({
    "Sample": val["Sample No."].values,

    "P_exp": y_val["P"].values,
    "P_pred": P,

    "N_exp": y_val["N"].values,
    "N_pred": N,

    "A_exp": y_val["A"].values,
    "A_pred": A
})

result.round(2)

,Sample,P_exp,P_pred,N_exp,N_pred,A_exp,A_pred
0,17,15.8,10.38,35.9,29.350000,48.3,60.270000
1,18,14.4,13.18,23.3,26.440001,62.2,60.380001
2,19,4.9,10.78,21.1,26.230000,74.0,62.990002
3,20,10.9,14.54,29.9,34.959999,59.2,50.500000
4,21,10.4,10.62,18.5,26.139999,71.0,63.240002
5,22,6.3,12.20,26.2,26.219999,67.5,61.580002


In [8]:
from sklearn.metrics import mean_squared_error, r2_score

obs = y_val[["P", "N", "A"]].values.ravel()
prd = pred.ravel()

rmse = np.sqrt(mean_squared_error(obs, prd))
r2 = r2_score(obs, prd)

print("RMSE =", round(rmse, 3))
print("R²   =", round(r2, 4))

RMSE = 6.306
R²   = 0.9265
